# 🎨 Artist Impersonator

Turn your personal photos into paintings in the style of 17 artists and ancient
art traditions — plus one **composite** that blends all of their styles together.

### How it works
The transform is done with **arbitrary neural style transfer** using Google's
*Magenta* `arbitrary-image-stylization-v1-256` model (from TensorFlow Hub). This
is a small generative network that was **trained on ~80,000 paintings** (the
*Painter by Numbers* / WikiArt corpus, which is dominated by Impressionist and
Post‑Impressionist work). At inference it looks at one *reference artwork*, reads
its brushwork/palette/texture statistics, and repaints your photo with them.

For each artist the notebook fetches a representative reference painting from
**Wikimedia Commons** (public‑domain works), caches it, and stylises every photo
with it. The **composite** re‑combines all the stylised versions of each photo
(mean / median / geometric‑mean of the pixels).

A second engine — classic **VGG‑19 optimisation style transfer** (Gatys et al.,
pure PyTorch) — is included as a fallback / higher‑fidelity option. There the
composite is a *single* optimisation driven by all the references at once.

### Usage
1. Put your photos in **`input_photos/`** (created on first run; a couple of
   sample photos are downloaded if it's empty).
2. Run all cells. Results land in **`output/<photo name>/`**:
   `Monet.jpg`, `VanGogh.jpg`, … , `_Composite.jpg`, `_contact_sheet.jpg`.
3. Tweak the **Config** cell (styles, strength, size, engine) and re‑run.

### Requirements
`numpy`, `pillow`, `matplotlib`, `requests` (all normally present) plus, per
engine, either `tensorflow` + `tensorflow_hub` (**magenta**, the default) or
`torch` (**vgg**). The Setup cell installs whatever the chosen engine is missing.

### A note on copyright
The Impressionists (Monet, Renoir, Degas, Pissarro, Sisley…) and the others up to
~1930 are public domain, so their reference paintings download automatically.
**Dalí, Picasso, Matisse and Xul Solar may still be under copyright** in your
country — for those, drop a reference image you have the right to use into
`style_references/<Artist>/` and the notebook will use it. If nothing is
available for an artist, that artist is skipped and left out of the composite.
The **Aztec** (an Aztec codex page) and **Egyptian** (a Book of the Dead papyrus)
references are photographs of works hundreds to thousands of years old and are
public domain worldwide.

## 1 · Config

In [ ]:
# ── Folders ──────────────────────────────────────────────────────────────────
INPUT_DIR   = "input_photos"      # your photos go here
OUTPUT_DIR  = "output"            # results are written here, one sub-folder per photo
STYLE_DIR   = "style_references"  # drop your own reference artworks in <Artist>/ sub-folders
CACHE_DIR   = ".cache"            # downloaded models + reference images

# ── Artists (order preserved; also the file/label names) ─────────────────────
ARTISTS = [
    "Aztec", "Cezanne", "Dali", "Degas", "Egyptian", "Gauguin", "Hokusai",
    "Manet", "Matisse", "Monet", "Picasso", "Pissarro", "Renoir", "Rodin",
    "Seurat", "Xul Solar", "vanGogh",
]

# ── Engine ──────────────────────────────────────────────────────────────────
ENGINE = "auto"          # "magenta" (fast, TF-Hub) | "vgg" (slow, PyTorch) | "auto"

# ── Output / quality ────────────────────────────────────────────────────────
OUTPUT_LONG_SIDE  = 1024        # longest side of every stylised image, px
STYLE_STRENGTH    = 1.00        # 0..1  (magenta: blend stylised<->photo; vgg: relative style weight)
STYLE_STRENGTH_OVERRIDES = {    # per-artist tweaks; e.g. dial a heavy style back toward the photo
    # "Hokusai": 0.9, "vanGogh": 0.9, "Rodin": 0.85,
}
COMPOSITE_METHOD  = "mean"      # "mean" | "median" | "geomean"  (magenta only; vgg optimises jointly)
COMPOSITE_AUTOCONTRAST = True

# ── Batch control ───────────────────────────────────────────────────────────
MAX_PHOTOS               = 0      # cap photos processed per run (None = all)
MAKE_CONTACT_SHEET       = True
DISPLAY_INLINE           = True
DOWNLOAD_SAMPLE_IF_EMPTY = True   # grab a few public-domain sample photos if input_photos/ is empty
OVERWRITE                = False  # re-render outputs that already exist?

# ── VGG engine only ─────────────────────────────────────────────────────────
VGG_SIZE    = 128      # working resolution for the optimisation (px, longest side)
VGG_STEPS   = 540      # L-BFGS iterations per image (more = stronger, slower)
VGG_DEVICE  = "auto"   # "auto" | "mps" | "cuda" | "cpu"

# ── Networking ─────────────────────────────────────────────────────────────-
# Wikimedia asks for a descriptive User-Agent with contact info.
WIKIMEDIA_UA = "Jupyter-notebook/1.0"

SEED = 0

## 2 · Setup — imports & dependencies

In [ ]:
import os, sys, io, math, time, subprocess, pathlib, warnings, urllib.parse
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # quiet TensorFlow
import numpy as np
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass
np.random.seed(SEED)

def _pip(*pkgs):
    print("  pip install", *pkgs, "…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import requests
except ImportError:
    _pip("requests"); import requests
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x=None, **k): return x if x is not None else []

for d in (INPUT_DIR, OUTPUT_DIR, STYLE_DIR, CACHE_DIR):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TFHUB_CACHE_DIR", str(pathlib.Path(CACHE_DIR, "tfhub").resolve()))

HAVE_TF = HAVE_TORCH = False
try:
    import tensorflow as tf; HAVE_TF = True
except Exception:
    pass
try:
    import torch; HAVE_TORCH = True
except Exception:
    pass


def _ensure_magenta():
    """Make `tensorflow` + `tensorflow_hub` importable; return the hub module."""
    global tf, HAVE_TF
    if not HAVE_TF:
        _pip("tensorflow"); import tensorflow as tf; HAVE_TF = True
    try:
        import pkg_resources          # tensorflow_hub imports this at load time
    except Exception:
        _pip("setuptools<81")         # newer setuptools dropped pkg_resources
    try:
        import tensorflow_hub as hub
    except Exception:
        _pip("tensorflow_hub", "setuptools<81")
        import tensorflow_hub as hub
    return hub


def _ensure_vgg():
    """Make `torch` importable; return it."""
    global torch, HAVE_TORCH
    if not HAVE_TORCH:
        _pip("torch"); import torch; HAVE_TORCH = True
    return torch


# ---- decide which engine to use --------------------------------------------
def _resolve_engine(choice):
    choice = (choice or "auto").lower()
    if choice == "magenta":
        return "magenta"
    if choice == "vgg":
        return "vgg"
    # auto: prefer magenta (fast + trained on paintings); fall back to vgg
    if HAVE_TF or not HAVE_TORCH:
        return "magenta"
    return "vgg"

ENGINE = _resolve_engine(ENGINE)
print("Engine:", ENGINE)
if ENGINE == "magenta":
    hub = _ensure_magenta()
    print("  tensorflow", tf.__version__)
else:
    torch = _ensure_vgg()
    print("  torch", torch.__version__)
print("Folders ready:", INPUT_DIR, OUTPUT_DIR, STYLE_DIR, CACHE_DIR)

## 3 · Style-reference registry

For each artist: known public-domain works on Wikimedia Commons (`files`, tried in
order) and text queries for the Commons search API (`search`, used if the direct
files fail). Anything you place in `style_references/<Artist>/` **overrides** both.

In [ ]:
# filename -> https://commons.wikimedia.org/wiki/Special:FilePath/<filename>
STYLE_REFERENCES = {
    "Aztec": dict(
        files=["Codex Borbonicus (p. 13).jpg",
               "Aztec Sun Stone or Calendar Stone.jpg"],
        search=["Aztec codex painting Mesoamerican", "Aztec calendar stone carving relief"]),
    "Cezanne": dict(
        files=["Paul_Cézanne_108.jpg",
               "Paul_Cézanne_-_Mont_Sainte-Victoire_-_Google_Art_Project.jpg"],
        search=["Paul Cezanne landscape painting", "Cezanne Mont Sainte-Victoire painting"]),
    "Dali": dict(
        files=[],
        search=['"Salvador Dali" painting', 'Salvador Dali surrealist oil painting canvas'],
        require=["dali", "dalí"],
        note="Dalí is under copyright — add your own image to style_references/Dali/."),
    "Degas": dict(
        files=["Edgar_Germain_Hilaire_Degas_024.jpg",
               "Edgar_Degas_-_The_Ballet_Class_-_Google_Art_Project.jpg"],
        search=["Edgar Degas ballet dancers painting", "Degas pastel dancers"]),
    "Egyptian": dict(
        files=["Book of the Dead of Hunefer sheet 3.jpg",
               "Tomb of Nebamun.jpg"],
        search=["ancient Egyptian tomb wall painting fresco", "Egyptian papyrus Book of the Dead"]),
    "Gauguin": dict(
        files=["Paul_Gauguin_105.jpg",
               "Paul_Gauguin_-_D'ou_venons-nous.jpg"],
        search=["Paul Gauguin Tahiti painting", "Gauguin post-impressionism painting"]),
    "Hokusai": dict(
        files=["Great_Wave_off_Kanagawa2.jpg",
               "Red_Fuji_southern_wind_clear_morning.jpg"],
        search=["Hokusai ukiyo-e woodblock print", "Katsushika Hokusai print landscape"]),
    "Manet": dict(
        files=["Edouard_Manet_-_Luncheon_on_the_Grass_-_Google_Art_Project.jpg"],
        search=["Edouard Manet painting", "Manet oil painting"]),
    "Matisse": dict(
        files=[],
        search=['"Henri Matisse" painting', "Matisse fauvism painting canvas"],
        require=["matisse"],
        note="Matisse may be under copyright where you are — add your own image to style_references/Matisse/."),
    "Monet": dict(
        files=["Claude_Monet,_Impression,_soleil_levant.jpg",
               "Claude_Monet_-_Water_Lilies_-_1906,_Ryerson.jpg"],
        search=["Claude Monet water lilies painting", "Monet impressionist landscape painting"]),
    "Picasso": dict(
        files=[],
        search=['"Pablo Picasso" painting', "Picasso cubism oil painting canvas"],
        require=["picasso"],
        note="Picasso is under copyright — add your own image to style_references/Picasso/."),
    "Pissarro": dict(
        files=["Camille_Pissarro_010.jpg",
               "Camille_Pissarro_-_Boulevard_Montmartre_-_Google_Art_Project.jpg"],
        search=["Camille Pissarro boulevard painting", "Pissarro impressionist street painting"]),
    "Renoir": dict(
        files=["Pierre-Auguste_Renoir,_Le_Moulin_de_la_Galette.jpg",
               "Pierre-Auguste_Renoir_-_Luncheon_of_the_Boating_Party_-_Google_Art_Project.jpg"],
        search=["Pierre-Auguste Renoir painting figures", "Renoir impressionist painting"]),
    "Rodin": dict(
        files=["The_Thinker,_Rodin.jpg"],
        search=["Auguste Rodin bronze sculpture", "Rodin marble sculpture"],
        note="Rodin was a sculptor — the transfer picks up bronze/marble tone and modelled form."),
    "Seurat": dict(
        files=["A_Sunday_on_La_Grande_Jatte,_Georges_Seurat,_1884.jpg",
               "Georges_Seurat_066.jpg"],
        search=["Georges Seurat pointillism painting", "Seurat La Grande Jatte painting"]),
    "Xul Solar": dict(
        files=[],
        search=['"Xul Solar" painting', '"Xul Solar" watercolour', "Xul Solar Alejandro Schulz artwork"],
        require=["xul solar", "schulz solari"],
        note="Xul Solar is under copyright — add your own image to style_references/Xul Solar/."),
    "vanGogh": dict(
        files=["Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg",
               "Vincent_van_Gogh_-_Wheatfield_with_crows_-_Google_Art_Project.jpg",
               "Vincent_Willem_van_Gogh_128.jpg"],
        search=["Vincent van Gogh painting", "Van Gogh post-impressionist painting"]),
}

FILEPATH = "https://commons.wikimedia.org/wiki/Special:FilePath/{}?width=1400"
COMMONS_API = "https://commons.wikimedia.org/w/api.php"
HTTP = requests.Session()
HTTP.headers.update({"User-Agent": WIKIMEDIA_UA})

## 4 · Download & cache the reference artworks

In [ ]:
def _fetch(url, timeout=60, tries=3):
    last = None
    for i in range(tries):
        try:
            r = HTTP.get(url, timeout=timeout); r.raise_for_status()
            return r.content
        except Exception as e:
            last = e; time.sleep(1.5 * (i + 1))
    raise last

def _valid_image(b, min_px=200):
    try:
        im = Image.open(io.BytesIO(b)); im.load()
        return min(im.size) >= min_px
    except Exception:
        return False

# titles that almost never denote a usable painting/print scan
_BAD_TITLE = ("atomicus", "facade", "fachada", "photograph", "photo of", " foto",
              "exterior", "street view", "signature", "grave", "tomb", "headstone",
              "juan gris", "exhibition", "installation view", "postage", "stamp",
              "banknote", "book cover", "plaque", "google map", "self-portrait photo",
              "gallery", "escuela", "school", "campus", "university", "conference",
              "wikimania", "meetup", "building", "mural at", "reproduction in")

def _commons_search(query, limit=15, require=None):
    """Return (image_url, title) for the best raster hit, or (None, None).

    require: optional list of lower-case substrings; the file title must contain
    at least one of them (used to keep results on the right artist).
    """
    p = dict(action="query", format="json", prop="imageinfo", generator="search",
             gsrsearch=query, gsrnamespace=6, gsrlimit=limit,
             iiprop="url|mime|size", iiurlwidth=1400)
    try:
        d = HTTP.get(COMMONS_API, params=p, timeout=60).json()
    except Exception:
        return None, None
    pages = sorted(d.get("query", {}).get("pages", {}).values(),
                   key=lambda x: x.get("index", 1e9))
    for pg in pages:
        ii = (pg.get("imageinfo") or [{}])[0]
        if ii.get("mime") not in ("image/jpeg", "image/png") or not ii.get("thumburl"):
            continue
        if ii.get("size", 0) and ii["size"] < 40_000:      # skip icons / tiny files
            continue
        t = pg.get("title", "").replace("File:", "")
        tl = t.lower()
        if any(b in tl for b in _BAD_TITLE):
            continue
        if require and not any(r in tl for r in require):
            continue
        return ii["thumburl"], t
    return None, None

def _load_local_refs(artist):
    d = pathlib.Path(STYLE_DIR) / artist
    out = []
    if d.is_dir():
        for p in sorted(d.iterdir()):
            if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"):
                try:
                    out.append(ImageOps.exif_transpose(Image.open(p)).convert("RGB"))
                except Exception:
                    pass
    return out

def ensure_style_refs(artists):
    """For every artist return a list of PIL reference images; report the source."""
    refs, report = {}, []
    for a in artists:
        spec = STYLE_REFERENCES.get(a, {})
        cache_a = pathlib.Path(CACHE_DIR) / "styles" / a
        cache_a.mkdir(parents=True, exist_ok=True)

        local = _load_local_refs(a)
        if local:
            refs[a] = local; report.append((a, f"your file(s) in {STYLE_DIR}/{a}/", len(local))); continue

        cimgs = []
        for p in sorted(cache_a.glob("*.jpg")):
            try: cimgs.append(Image.open(p).convert("RGB"))
            except Exception: pass
        if cimgs:
            refs[a] = cimgs; report.append((a, "cache", len(cimgs))); continue

        got, src, guess = [], None, False
        for fn in spec.get("files", []):
            try:
                b = _fetch(FILEPATH.format(urllib.parse.quote(fn)))
                if _valid_image(b):
                    got.append(b); src = "Wikimedia Commons (curated)"
                    break
            except Exception:
                continue
        if not got:
            for q in spec.get("search", []):
                u, title = _commons_search(q, require=spec.get("require"))
                if u:
                    try:
                        b = _fetch(u)
                        if _valid_image(b):
                            got.append(b); src = f"Commons search · {title}"
                            guess = bool(spec.get("note"))
                            break
                    except Exception:
                        continue
        if got:
            imgs = []
            for i, b in enumerate(got):
                im = ImageOps.exif_transpose(Image.open(io.BytesIO(b))).convert("RGB")
                im.save(cache_a / f"ref{i}.jpg", quality=95)
                imgs.append(im)
            refs[a] = imgs
            report.append((a, ("GUESS " if guess else "OK ") + str(len(imgs)), src))
        else:
            refs[a] = []
            hint = "add an image to %s/%s/" % (STYLE_DIR, a)
            report.append((a, "MISSING", "— " + (spec.get("note") or hint)))
    # pretty report
    print("Reference artworks")
    print("-" * 74)
    for a, status, src in report:
        print(f"  {a:<11} {status:<9} {src}")
    guessed = [a for a, s, _ in report if s.startswith("GUESS")]
    missing = [a for a, s, _ in report if s == "MISSING"]
    if guessed:
        print("\n  Auto-picked (verify / replace with your own in %s/<Artist>/):" % STYLE_DIR)
        print("   ", ", ".join(guessed))
    if missing:
        print("\n  Skipped — no reference available:", ", ".join(missing))
    return refs

STYLE_IMAGES = ensure_style_refs(ARTISTS)
ACTIVE_ARTISTS = [a for a in ARTISTS if STYLE_IMAGES.get(a)]
if not ACTIVE_ARTISTS:
    raise RuntimeError(
        "No style references available. Check your connection, or drop artwork "
        "images into %s/<Artist>/ folders and re-run." % STYLE_DIR)
print(f"\n{len(ACTIVE_ARTISTS)} styles ready:", ", ".join(ACTIVE_ARTISTS))

In [ ]:
# Preview the reference artworks
n = len(ACTIVE_ARTISTS)
cols = 5; rows = max(1, math.ceil(n / cols))
fig, ax = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.6), squeeze=False)
axes = ax.ravel()
for i, a in enumerate(ACTIVE_ARTISTS):
    axes[i].imshow(STYLE_IMAGES[a][0]); axes[i].set_title(a, fontsize=10)
for j in range(rows * cols):
    axes[j].axis("off")
plt.suptitle("Style references", y=1.002, fontsize=13); plt.tight_layout(); plt.show()

## 5 · Collect the input photos

In [ ]:
_IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".heic")

def _sample_photos():
    """A few scenic public-domain photos so the notebook runs with zero setup."""
    queries = ["Positano Amalfi Coast Italy photograph",
               "Neuschwanstein castle autumn photograph",
               "Golden Gate Bridge fog photograph",
               "portrait woman natural light photograph creative commons"]
    saved = []
    for q in queries:
        u, title = _commons_search(q, limit=15)
        if not u:
            continue
        try:
            b = _fetch(u)
            if not _valid_image(b, min_px=600):
                continue
            name = "sample_" + "".join(c if c.isalnum() else "_" for c in title.replace("File:", ""))[:48] + ".jpg"
            dst = pathlib.Path(INPUT_DIR) / name
            ImageOps.exif_transpose(Image.open(io.BytesIO(b))).convert("RGB").save(dst, quality=92)
            saved.append(dst)
        except Exception:
            continue
        if len(saved) >= 3:
            break
    return saved

photos = sorted(p for p in pathlib.Path(INPUT_DIR).iterdir()
                if p.is_file() and p.suffix.lower() in _IMG_EXT)
if not photos and DOWNLOAD_SAMPLE_IF_EMPTY:
    print(f"No photos in {INPUT_DIR}/ — downloading a few public-domain samples "
          f"(replace them with your own and re-run).")
    _sample_photos()
    photos = sorted(p for p in pathlib.Path(INPUT_DIR).iterdir()
                    if p.is_file() and p.suffix.lower() in _IMG_EXT)

if MAX_PHOTOS:
    photos = photos[:MAX_PHOTOS]
print(f"{len(photos)} photo(s) to process:")
for p in photos:
    print("  ", p.name)
assert photos, f"Put some images in {INPUT_DIR}/ and re-run."

## 6 · Style-transfer engine

Both engines expose the same two calls:

* `stylize_one(photo, [reference]) -> HxWx3 float array in [0, 1]`
* `stylize_composite(photo, [references], per_artist_results) -> array`

In [ ]:
def _pil(a):  # float array [0,1] -> PIL
    return Image.fromarray(np.clip(np.asarray(a) * 255, 0, 255).astype("uint8"))

def _resize_long(pil, long_side):
    im = ImageOps.exif_transpose(pil).convert("RGB")
    w, h = im.size
    s = long_side / max(w, h)
    return im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)

def _center_square(pil, n):
    im = ImageOps.exif_transpose(pil).convert("RGB")
    w, h = im.size; s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    return im.resize((n, n), Image.LANCZOS)

def _combine(arrs, method):
    st = np.stack(arrs, 0).astype("float32")
    if method == "median":
        out = np.median(st, 0)
    elif method == "geomean":
        out = np.exp(np.mean(np.log(np.clip(st, 1e-4, 1.0)), 0))
    else:
        out = st.mean(0)
    out = np.clip(out, 0, 1)
    if COMPOSITE_AUTOCONTRAST:
        out = np.asarray(ImageOps.autocontrast(_pil(out), cutoff=1), np.float32) / 255.0
    return out


class MagentaEngine:
    """Google Magenta arbitrary-image-stylization-v1-256 (TensorFlow Hub)."""
    URL = "https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2"

    def __init__(self):
        print("Loading Magenta style-transfer model …")
        t = time.time()
        self.model = hub.load(self.URL)
        print(f"  ready in {time.time()-t:.1f}s")

    def _run(self, content_pil, style_pil):
        c = np.asarray(_resize_long(content_pil, OUTPUT_LONG_SIDE), np.float32)[None] / 255.0
        s = np.asarray(_center_square(style_pil, 256), np.float32)[None] / 255.0
        out = self.model(tf.constant(c), tf.constant(s))[0].numpy()[0]
        return np.clip(out, 0, 1), c[0]

    def stylize_one(self, content_pil, style_pils, strength=1.0):
        out, base = self._run(content_pil, style_pils[0])
        if strength < 1.0:
            out = strength * out + (1 - strength) * base
        return np.clip(out, 0, 1)

    def stylize_composite(self, content_pil, style_pils, per_artist_results):
        # reuse the already-computed per-artist images
        return _combine(list(per_artist_results.values()), COMPOSITE_METHOD)


def _vgg19_features():
    import torch.nn as nn
    cfg = [64, 64, "M", 128, 128, "M", 256, 256, 256, 256, "M",
           512, 512, 512, 512, "M", 512, 512, 512, 512, "M"]
    layers, c = [], 3
    for v in cfg:
        if v == "M":
            layers.append(nn.MaxPool2d(2, 2))
        else:
            layers += [nn.Conv2d(c, v, 3, padding=1), nn.ReLU(inplace=True)]; c = v
    return nn.Sequential(*layers)


class VGGEngine:
    """Gatys et al. optimisation style transfer, pure PyTorch (no torchvision)."""
    WEIGHTS = "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth"
    STYLE_LAYERS   = [1, 6, 11, 20, 29]   # relu1_1 relu2_1 relu3_1 relu4_1 relu5_1
    CONTENT_LAYERS = [22]                  # relu4_2

    def __init__(self):
        want = VGG_DEVICE
        if want == "auto":
            want = ("mps" if torch.backends.mps.is_available()
                    else "cuda" if torch.cuda.is_available() else "cpu")
        self.device = torch.device(want)
        print(f"Building VGG-19 on {self.device} …")
        net = _vgg19_features()
        sd = torch.hub.load_state_dict_from_url(self.WEIGHTS, progress=True,
                                                model_dir=os.path.join(CACHE_DIR, "torch"))
        net.load_state_dict({k[len("features."):]: v for k, v in sd.items()
                             if k.startswith("features.")})
        net.eval()
        for p in net.parameters():
            p.requires_grad_(False)
        def _probe():
            x = torch.rand(1, 3, 24, 24, device=self.device, requires_grad=True)
            opt = torch.optim.LBFGS([x], max_iter=2, line_search_fn="strong_wolfe")
            def cl():
                opt.zero_grad()
                loss = (self._feats(x, [1])[1] ** 2).mean()
                loss.backward()
                return loss
            opt.step(cl)                                   # conv + autograd + optimiser
        try:
            self.net = net.to(self.device)
            _probe()
        except Exception as e:
            print(f"  {self.device} not usable for the optimiser ({type(e).__name__}); using CPU")
            self.device = torch.device("cpu"); self.net = net.to(self.device)
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=self.device).view(1, 3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225], device=self.device).view(1, 3, 1, 1)

    def _prep(self, pil, size):
        im = _resize_long(pil, size)
        x = torch.from_numpy(np.asarray(im, np.float32) / 255.0).permute(2, 0, 1)[None]
        return ((x.to(self.device) - self.mean) / self.std).contiguous()

    def _feats(self, x, idxs):
        want, hi, out, h = set(idxs), max(idxs), {}, x
        for i, layer in enumerate(self.net):
            h = layer(h)
            if i in want:
                out[i] = h
            if i >= hi:
                break
        return out

    @staticmethod
    def _gram(f):
        c, hh, ww = f.shape[1:]
        F = f.view(c, hh * ww)
        return (F @ F.t()) / (c * hh * ww)

    def _optimise(self, content_pil, style_pils, size, steps, style_weight):
        import torch.nn.functional as F
        content = self._prep(content_pil, size)
        with torch.no_grad():
            c_feats = {k: v.detach() for k, v in self._feats(content, self.CONTENT_LAYERS).items()}
            grams = []
            for sp in style_pils:
                sf = self._feats(self._prep(sp, size), self.STYLE_LAYERS)
                grams.append({k: self._gram(v).detach() for k, v in sf.items()})
        img = content.clone().detach().contiguous().requires_grad_(True)
        img.register_hook(lambda g: g.contiguous())        # LBFGS needs a contiguous grad
        opt = torch.optim.LBFGS([img], max_iter=steps, tolerance_grad=-1,
                                tolerance_change=-1, line_search_fn="strong_wolfe")
        allk = sorted(set(self.STYLE_LAYERS) | set(self.CONTENT_LAYERS))
        n = [0]
        def closure():
            opt.zero_grad()
            f = self._feats(img, allk)
            c_loss = sum(F.mse_loss(f[k], c_feats[k]) for k in self.CONTENT_LAYERS)
            s_loss = 0.0
            for g in grams:
                s_loss = s_loss + sum(F.mse_loss(self._gram(f[k]), g[k]) for k in self.STYLE_LAYERS)
            s_loss = s_loss / len(grams)
            loss = c_loss + style_weight * s_loss
            loss.backward()
            n[0] += 1
            if n[0] % 40 == 0:
                print(f"      iter {n[0]:4d}  loss={loss.item():.1f}")
            return loss
        opt.step(closure)
        with torch.no_grad():
            out = (img * self.std + self.mean).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
        # upscale from working size to the requested output size
        return np.asarray(_resize_long(_pil(out), OUTPUT_LONG_SIDE), np.float32) / 255.0

    def stylize_one(self, content_pil, style_pils, strength=1.0):
        # strength scales the style weight (log-spaced): 1.0 -> base, 0.5 -> /10
        sw = 1e6 * (10.0 ** (2 * (strength - 1.0)))
        return self._optimise(content_pil, style_pils, VGG_SIZE, VGG_STEPS, sw)

    def stylize_composite(self, content_pil, style_pils, per_artist_results):
        return self._optimise(content_pil, style_pils, VGG_SIZE,
                              max(VGG_STEPS, 300), 1e6)


engine = MagentaEngine() if ENGINE == "magenta" else VGGEngine()

## 7 · Transform every photo

In [ ]:
def contact_sheet(original, tiles, ncol=6, tile=360, pad=10, title=""):
    """tiles: list of (label, PIL). Returns a labelled grid PIL."""
    items = [("Original", original)] + tiles
    nrow = math.ceil(len(items) / ncol)
    W = ncol * tile + (ncol + 1) * pad
    head = 34 if title else 0
    H = head + nrow * (tile + 22) + pad
    sheet = Image.new("RGB", (W, H), "white")
    d = ImageDraw.Draw(sheet)
    if title:
        d.text((pad, 9), title, fill="black")
    for i, (label, im) in enumerate(items):
        r, c = divmod(i, ncol)
        x = pad + c * (tile + pad)
        y = head + pad + r * (tile + 22)
        t = im.copy(); t.thumbnail((tile, tile), Image.LANCZOS)
        sheet.paste(t, (x + (tile - t.width) // 2, y + (tile - t.height) // 2))
        d.text((x + 2, y + tile + 4), label, fill="black")
    return sheet

t_all = time.time()
for pi, ppath in enumerate(photos, 1):
    photo = ImageOps.exif_transpose(Image.open(ppath)).convert("RGB")
    stem = ppath.stem
    odir = pathlib.Path(OUTPUT_DIR) / stem
    odir.mkdir(parents=True, exist_ok=True)
    print(f"\n[{pi}/{len(photos)}] {ppath.name}  ({photo.size[0]}x{photo.size[1]})")

    results, tiles = {}, []
    for a in ACTIVE_ARTISTS:
        dst = odir / f"{a.replace(' ', '')}.jpg"
        strength = STYLE_STRENGTH_OVERRIDES.get(a, STYLE_STRENGTH)
        if dst.exists() and not OVERWRITE:
            arr = np.asarray(Image.open(dst).convert("RGB"), np.float32) / 255.0
        else:
            t = time.time()
            arr = engine.stylize_one(photo, STYLE_IMAGES[a], strength)
            _pil(arr).save(dst, quality=93)
            print(f"   {a:<11} {time.time()-t:5.1f}s -> {dst.name}")
        results[a] = arr
        tiles.append((a, _pil(arr)))

    cdst = odir / "_Composite.jpg"
    if cdst.exists() and not OVERWRITE:
        comp = np.asarray(Image.open(cdst).convert("RGB"), np.float32) / 255.0
    else:
        t = time.time()
        flat_refs = [im for a in ACTIVE_ARTISTS for im in STYLE_IMAGES[a][:1]]
        comp = engine.stylize_composite(photo, flat_refs, results)
        _pil(comp).save(cdst, quality=93)
        print(f"   {'Composite':<11} {time.time()-t:5.1f}s -> {cdst.name}  ({COMPOSITE_METHOD})")
    tiles.append(("Composite", _pil(comp)))

    if MAKE_CONTACT_SHEET:
        sheet = contact_sheet(photo, tiles, title=f"{ppath.name}   ·   engine: {ENGINE}")
        sheet.save(odir / "_contact_sheet.jpg", quality=90)
        if DISPLAY_INLINE:
            plt.figure(figsize=(16, 16 * sheet.height / sheet.width))
            plt.imshow(sheet); plt.axis("off"); plt.title(ppath.name); plt.show()

print(f"\nAll done in {time.time()-t_all:.0f}s. Results in ./{OUTPUT_DIR}/")

## 8 · Tips & extensions

**Dial a style in or out** — `STYLE_STRENGTH` (global) or `STYLE_STRENGTH_OVERRIDES`
(per artist). With the *magenta* engine this blends the painting back toward the
photo; with *vgg* it scales the style weight.

**Sharper / larger output** — raise `OUTPUT_LONG_SIDE` (magenta re-runs at the new
size for free). For the *vgg* engine also raise `VGG_SIZE` and `VGG_STEPS`.

**Swap the reference painting** — drop your preferred image into
`style_references/<Artist>/` (e.g. a different Monet). It overrides the download.
Multiple files there: *magenta* uses the first, *vgg* blends all of them.

**Composite flavour** — `COMPOSITE_METHOD`: `"mean"` (soft, balanced),
`"median"` (crisper, rejects outlier styles), `"geomean"` (richer, higher
contrast). *vgg* ignores this and instead optimises one image against every
reference at once.

**Speed** — *magenta* is ~1 s per image on CPU. *vgg* is minutes per image; it
uses Apple‑Silicon `mps` / CUDA automatically when available. Cut `MAX_PHOTOS`
while experimenting; finished outputs are skipped on re‑runs unless `OVERWRITE`.

**Going further**
* *True per‑artist GANs* — Berkeley's CycleGAN ships models trained end‑to‑end on
  one artist's body of work (`style_monet`, `style_cezanne`, `style_ukiyoe`,
  `style_vangogh`). Higher fidelity, but only those four.
* *Diffusion* — SDXL / SD img2img with a prompt like *"in the style of Claude
  Monet, impressionist oil painting"* and low denstrength gives a different,
  more interpretive result; needs a GPU or an API key.